In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
student_info = pd.read_csv("studentInfo.csv")
student_assessment = pd.read_csv("studentAssessment.csv")
assessments = pd.read_csv("assessments.csv")
student_vle = pd.read_csv("studentVle.csv")

print("student_info:", student_info.shape)
print("student_assessment:", student_assessment.shape)
print("assessments:", assessments.shape)
print("student_vle:", student_vle.shape)

student_info: (32593, 12)
student_assessment: (173912, 5)
assessments: (206, 6)
student_vle: (10655280, 6)


In [3]:
data_base = student_info.copy()

data_base["imd_band"] = data_base["imd_band"].fillna("Unknown")

data_base["at_risk"] = data_base["final_result"].apply(
    lambda x: 1 if x in ["Fail", "Withdrawn"] else 0
)

print(data_base["at_risk"].value_counts())
print(data_base["at_risk"].value_counts(normalize=True) * 100)

at_risk
1    17208
0    15385
Name: count, dtype: int64
at_risk
1    52.796613
0    47.203387
Name: proportion, dtype: float64


In [4]:
early_day_cutoff = 120

assessment_merged = student_assessment.merge(
    assessments,
    on="id_assessment",
    how="left"
)

# Remove rows without score
assessment_merged = assessment_merged.dropna(subset=["score"]).copy()

# Remove Exam to reduce temporal leakage
assessment_merged_no_exam = assessment_merged[
    assessment_merged["assessment_type"] != "Exam"
].copy()

# Keep only early assessments
assessment_early = assessment_merged_no_exam[
    assessment_merged_no_exam["date"] <= early_day_cutoff
].copy()

print("assessment_early shape:", assessment_early.shape)
print(assessment_early["assessment_type"].value_counts())
print("Max date:", assessment_early["date"].max())

assessment_early shape: (79936, 10)
assessment_type
TMA    62606
CMA    17330
Name: count, dtype: int64
Max date: 117.0


In [5]:
assessment_early["submitted_late"] = (
    assessment_early["date_submitted"] > assessment_early["date"]
).astype(int)

assessment_early["submission_delay"] = (
    assessment_early["date_submitted"] - assessment_early["date"]
)

assessment_early["weighted_score"] = (
    assessment_early["score"] * assessment_early["weight"]
)

assessment_features_early = assessment_early.groupby(
    ["id_student", "code_module", "code_presentation"]
).agg(
    early_avg_score=("score", "mean"),
    early_max_score=("score", "max"),
    early_min_score=("score", "min"),
    early_score_std=("score", "std"),
    early_submitted_assessments_count=("id_assessment", "count"),
    early_late_submissions_count=("submitted_late", "sum"),
    early_avg_submission_delay=("submission_delay", "mean"),
    early_total_weight_attempted=("weight", "sum"),
    early_total_weighted_score=("weighted_score", "sum")
).reset_index()

assessment_features_early["early_weighted_score_avg"] = (
    assessment_features_early["early_total_weighted_score"] /
    assessment_features_early["early_total_weight_attempted"]
)

assessment_features_early["early_late_submission_rate"] = (
    assessment_features_early["early_late_submissions_count"] /
    assessment_features_early["early_submitted_assessments_count"]
)

assessment_features_early["early_weighted_score_avg"] = assessment_features_early[
    "early_weighted_score_avg"
].replace([np.inf, -np.inf], np.nan)

assessment_features_early["early_weighted_score_avg"] = assessment_features_early[
    "early_weighted_score_avg"
].fillna(assessment_features_early["early_avg_score"])

assessment_features_early["early_score_std"] = assessment_features_early["early_score_std"].fillna(0)

print("assessment_features_early:", assessment_features_early.shape)
display(assessment_features_early.head())
print(assessment_features_early.isnull().sum())

assessment_features_early: (25724, 14)


,id_student,code_module,code_presentation,early_avg_score,early_max_score,early_min_score,early_score_std,early_submitted_assessments_count,early_late_submissions_count,early_avg_submission_delay,early_total_weight_attempted,early_total_weighted_score,early_weighted_score_avg,early_late_submission_rate
0,6516,AAA,2014J,57.000000,63.0,48.0,7.937254,3,0,-2.000000,50.0,2820.0,56.40,0.000000
1,8462,DDD,2013J,87.666667,93.0,83.0,5.033223,3,1,-0.333333,40.0,3490.0,87.25,0.333333
2,8462,DDD,2014J,86.500000,93.0,83.0,4.725816,4,0,-59.500000,50.0,4300.0,86.00,0.000000
3,11391,AAA,2013J,81.000000,85.0,78.0,3.605551,3,0,-1.333333,50.0,4080.0,81.60,0.000000
4,23629,BBB,2013B,82.500000,100.0,63.0,20.273135,4,3,3.500000,25.0,1669.0,66.76,0.750000


id_student                           0
code_module                          0
code_presentation                    0
early_avg_score                      0
early_max_score                      0
early_min_score                      0
early_score_std                      0
early_submitted_assessments_count    0
early_late_submissions_count         0
early_avg_submission_delay           0
early_total_weight_attempted         0
early_total_weighted_score           0
early_weighted_score_avg             0
early_late_submission_rate           0
dtype: int64


In [6]:
print("student_vle shape:", student_vle.shape)
display(student_vle.head())
print(student_vle.info())
print(student_vle.isnull().sum())

student_vle shape: (10655280, 6)


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10655280 entries, 0 to 10655279
Data columns (total 6 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   code_module        object
 1   code_presentation  object
 2   id_student         int64 
 3   id_site            int64 
 4   date               int64 
 5   sum_click          int64 
dtypes: int64(4), object(2)
memory usage: 487.8+ MB
None
code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64


In [7]:
early_day_cutoff = 60

student_vle_early = student_vle[
    student_vle["date"] <= early_day_cutoff
].copy()

print("Original student_vle:", student_vle.shape)
print("Early student_vle:", student_vle_early.shape)

print("Min date:", student_vle_early["date"].min())
print("Max date:", student_vle_early["date"].max())

display(student_vle_early.head())

Original student_vle: (10655280, 6)
Early student_vle: (4499256, 6)
Min date: -25
Max date: 60


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


In [8]:
vle_features_early = student_vle_early.groupby(
    ["id_student", "code_module", "code_presentation"]
).agg(
    early_total_clicks=("sum_click", "sum"),
    early_avg_clicks=("sum_click", "mean"),
    early_max_clicks=("sum_click", "max"),
    early_clicks_std=("sum_click", "std"),
    early_active_days=("date", "nunique"),
    early_unique_sites=("id_site", "nunique"),
    early_first_activity_day=("date", "min"),
    early_last_activity_day=("date", "max")
).reset_index()

vle_features_early["early_clicks_per_active_day"] = (
    vle_features_early["early_total_clicks"] /
    vle_features_early["early_active_days"]
)

vle_features_early["early_activity_span"] = (
    vle_features_early["early_last_activity_day"] -
    vle_features_early["early_first_activity_day"]
)

vle_features_early["early_clicks_std"] = vle_features_early["early_clicks_std"].fillna(0)

print("vle_features_early shape:", vle_features_early.shape)
display(vle_features_early.head())
print(vle_features_early.isnull().sum())

vle_features_early shape: (29109, 13)


,id_student,code_module,code_presentation,early_total_clicks,early_avg_clicks,early_max_clicks,early_clicks_std,early_active_days,early_unique_sites,early_first_activity_day,early_last_activity_day,early_clicks_per_active_day,early_activity_span
0,6516,AAA,2014J,1046,4.733032,32,5.760070,48,44,-23,60,21.791667,83
1,8462,DDD,2013J,523,2.355856,16,2.526679,37,91,-6,60,14.135135,66
2,8462,DDD,2014J,10,2.500000,7,3.000000,1,3,10,10,10.000000,0
3,11391,AAA,2013J,529,5.877778,76,10.053595,18,33,-5,54,29.388889,59
4,23629,BBB,2013B,119,2.767442,13,2.776064,12,9,-6,59,9.916667,65


id_student                     0
code_module                    0
code_presentation              0
early_total_clicks             0
early_avg_clicks               0
early_max_clicks               0
early_clicks_std               0
early_active_days              0
early_unique_sites             0
early_first_activity_day       0
early_last_activity_day        0
early_clicks_per_active_day    0
early_activity_span            0
dtype: int64


In [9]:
data_v3 = data_base.copy()

data_v3 = data_v3.merge(
    assessment_features_early,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

data_v3 = data_v3.merge(
    vle_features_early,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

print("data_v3 shape:", data_v3.shape)
display(data_v3.head())
print(data_v3.isnull().sum())

data_v3 shape: (32593, 34)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,early_total_clicks,early_avg_clicks,early_max_clicks,early_clicks_std,early_active_days,early_unique_sites,early_first_activity_day,early_last_activity_day,early_clicks_per_active_day,early_activity_span
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,529.0,5.877778,76.0,10.053595,18.0,33.0,-5.0,54.0,29.388889,59.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,669.0,3.716667,19.0,3.777728,26.0,34.0,-10.0,53.0,25.730769,63.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,281.0,3.697368,23.0,4.056336,12.0,22.0,-10.0,12.0,23.416667,22.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,816.0,3.303644,22.0,3.287218,41.0,38.0,-10.0,60.0,19.902439,70.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,605.0,3.757764,22.0,3.946163,28.0,35.0,-10.0,60.0,21.607143,70.0


code_module                             0
code_presentation                       0
id_student                              0
gender                                  0
region                                  0
highest_education                       0
imd_band                                0
age_band                                0
num_of_prev_attempts                    0
studied_credits                         0
disability                              0
final_result                            0
at_risk                                 0
early_avg_score                      6869
early_max_score                      6869
early_min_score                      6869
early_score_std                      6869
early_submitted_assessments_count    6869
early_late_submissions_count         6869
early_avg_submission_delay           6869
early_total_weight_attempted         6869
early_total_weighted_score           6869
early_weighted_score_avg             6869
early_late_submission_rate        

In [10]:
early_assessment_numeric_features = [
    "early_avg_score",
    "early_max_score",
    "early_min_score",
    "early_score_std",
    "early_submitted_assessments_count",
    "early_late_submissions_count",
    "early_avg_submission_delay",
    "early_total_weight_attempted",
    "early_total_weighted_score",
    "early_weighted_score_avg",
    "early_late_submission_rate"
]

early_vle_numeric_features = [
    "early_total_clicks",
    "early_avg_clicks",
    "early_max_clicks",
    "early_clicks_std",
    "early_active_days",
    "early_unique_sites",
    "early_first_activity_day",
    "early_last_activity_day",
    "early_clicks_per_active_day",
    "early_activity_span"
]

data_v3[early_assessment_numeric_features] = data_v3[early_assessment_numeric_features].fillna(0)
data_v3[early_vle_numeric_features] = data_v3[early_vle_numeric_features].fillna(0)

print(data_v3[early_assessment_numeric_features + early_vle_numeric_features].isnull().sum())

early_avg_score                      0
early_max_score                      0
early_min_score                      0
early_score_std                      0
early_submitted_assessments_count    0
early_late_submissions_count         0
early_avg_submission_delay           0
early_total_weight_attempted         0
early_total_weighted_score           0
early_weighted_score_avg             0
early_late_submission_rate           0
early_total_clicks                   0
early_avg_clicks                     0
early_max_clicks                     0
early_clicks_std                     0
early_active_days                    0
early_unique_sites                   0
early_first_activity_day             0
early_last_activity_day              0
early_clicks_per_active_day          0
early_activity_span                  0
dtype: int64


In [11]:
base_features = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability"
]

features_v3 = base_features + early_assessment_numeric_features + early_vle_numeric_features

X = data_v3[features_v3]
y = data_v3["at_risk"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (32593, 31)
y shape: (32593,)


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Train distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("Test distribution:")
print(y_test.value_counts(normalize=True) * 100)

Train shape: (26074, 31)
Test shape: (6519, 31)
Train distribution:
at_risk
1    52.795889
0    47.204111
Name: proportion, dtype: float64
Test distribution:
at_risk
1    52.799509
0    47.200491
Name: proportion, dtype: float64


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability"
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits"
] + early_assessment_numeric_features + early_vle_numeric_features

preprocessor_v3 = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

xgb_model_v3 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v3),
        ("model", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ]
)

xgb_model_v3.fit(X_train, y_train)

y_pred_xgb_v3 = xgb_model_v3.predict(X_test)

print("XGBoost V3 Accuracy:", accuracy_score(y_test, y_pred_xgb_v3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb_v3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb_v3))

XGBoost V3 Accuracy: 0.8677711305414941

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      3077
           1       0.91      0.83      0.87      3442

    accuracy                           0.87      6519
   macro avg       0.87      0.87      0.87      6519
weighted avg       0.87      0.87      0.87      6519


Confusion Matrix:
[[2811  266]
 [ 596 2846]]


In [15]:
from sklearn.ensemble import RandomForestClassifier

rf_model_v3 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v3),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

rf_model_v3.fit(X_train, y_train)

y_pred_rf_v3 = rf_model_v3.predict(X_test)

print("Random Forest V3 Accuracy:", accuracy_score(y_test, y_pred_rf_v3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_v3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_v3))

Random Forest V3 Accuracy: 0.867310937260316

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.91      0.87      3077
           1       0.91      0.83      0.87      3442

    accuracy                           0.87      6519
   macro avg       0.87      0.87      0.87      6519
weighted avg       0.87      0.87      0.87      6519


Confusion Matrix:
[[2808  269]
 [ 596 2846]]
